# C7. The limit $\kappa\to0$, checked on solutions

Companion notebook to the bachelor's thesis *El formalismo ODM en sistemas híbridos clásico-cuánticos* (Santiago Puyol Miano, Universidad de Zaragoza, 2026).

In the variables $u=x-\tfrac{\hbar\kappa}{2}\lambda_p$ and $v=x+\tfrac{\hbar\kappa}{2}\lambda_p$, the second-order part of the evolution equation in the von Neumann representation has coefficient $(\hbar\kappa)^2/2m$, which vanishes as $\kappa\to0$. That alone says nothing about the solutions: $(u,v)$ depend on $\kappa$, the Jacobian of the inverse change of variables blows up in the limit, and losing the top-order term is the typical form of a singular perturbation.

In the κ-independent variables $(x,\lambda_p)$ the equation reads

$$i\,\partial_t\rho=\hat L\rho+M_\kappa\rho,\qquad \hat L=\tfrac1m\,\partial_x\partial_{\lambda_p}-U'(x)\,\lambda_p,$$

where $\hat L$ is the Liouvillian in the $(x,\lambda_p)$ representation and $M_\kappa$ is multiplication by

$$m_\kappa=\frac{U(x-s)-U(x+s)}{\hbar\kappa}+\lambda_pU'(x),\qquad s=\tfrac{\hbar\kappa}{2}\lambda_p,\qquad |m_\kappa|\le\frac{(\hbar\kappa)^2}{24}\,|\lambda_p|^3\sup|U'''|.$$

This notebook checks

1. the identity $\frac{(\hbar\kappa)^2}{2m}(\partial_v^2-\partial_u^2)=\frac{\hbar\kappa}{m}\partial_x\partial_{\lambda_p}$, so the principal part does not depend on $\kappa$ in the variables $(x,\lambda_p)$ and the order of the equation does not drop (symbolic);
2. $m_\kappa=-\frac{(\hbar\kappa)^2}{24}\lambda_p^3U'''(x)+O(\kappa^4)$, and $m_\kappa=0$ for every quadratic $U$, as in C4 (symbolic);
3. the bound on $|m_\kappa|$ on a grid for $U=-\cos x$ (numerical);
4. convergence of solutions: evolving the equation by Strang split-step Fourier for $U=-\cos x$, the $L^2$ distance at $t=1$ between the solution for $\kappa$ and the Liouville solution ($\kappa=0$) decreases at the rate $O(\kappa^2)$ when $\kappa$ is halved (numerical).

**Kernel:** Python 3 with sympy and numpy.

## 1. The principal part in the variables $(x,\lambda_p)$

In [1]:
import sympy as sp
import numpy as np

u, v, m_sym, hk = sp.symbols('u v m hbar_kappa', positive=True)
a, b = sp.symbols('a b')  # generic exponents

# Both sides are constant-coefficient linear operators after the linear change
# of variables, so equality on all exponentials G = e^{a x + b lp} with symbolic
# (a, b) is equality of their symbols, hence of the operators.
x_of = (u + v) / 2
lp_of = (v - u) / hk
rho = sp.exp(a * x_of + b * lp_of)  # G(x,lp) = e^{a x + b lp} composed with (u,v)

lhs = hk**2 / (2 * m_sym) * (sp.diff(rho, v, 2) - sp.diff(rho, u, 2))

# (hk/m) G_{x,lp} at the same point = (hk/m) * a * b * G
rhs = hk / m_sym * a * b * rho

assert sp.simplify(lhs - rhs) == 0, \
    f"((hk)^2/2m)(d_v^2-d_u^2) != (hk/m) d_x d_lp: {sp.simplify(lhs - rhs)}"
print("OK: ((hk)^2/2m)(d_v^2 - d_u^2) = (hk/m) d_x d_lp, so the principal part "
      "in (x, lambda_p) does not depend on kappa")

OK: ((hk)^2/2m)(d_v^2 - d_u^2) = (hk/m) d_x d_lp, so the principal part in (x, lambda_p) does not depend on kappa


## 2. Taylor expansion of $m_\kappa$

In [2]:
xs, lp, hbar, kap = sp.symbols('x lambda_p hbar kappa', real=True)
U = sp.Function('U')

s = hbar * kap * lp / 2
m_kappa = (U(xs - s) - U(xs + s)) / (hbar * kap) + lp * sp.diff(U(xs), xs)

series = sp.series(m_kappa, kap, 0, 4).removeO()
expected_leading = -(hbar * kap)**2 * lp**3 * sp.diff(U(xs), xs, 3) / 24
assert sp.simplify(series - expected_leading) == 0, \
    f"m_kappa expansion != -(hk)^2 lp^3 U'''/24 + O(k^4): {sp.simplify(series - expected_leading)}"
print("OK: m_kappa = -((hbar kappa)^2/24) lp^3 U'''(x) + O(kappa^4)")

# quadratic U: exact cancellation
a2, b2, c2 = sp.symbols('a2 b2 c2', real=True)
Uquad = a2 * xs**2 + b2 * xs + c2
m_kappa_quad = (Uquad.subs(xs, xs - s) - Uquad.subs(xs, xs + s)) / (hbar * kap) \
    + lp * sp.diff(Uquad, xs)
assert sp.expand(m_kappa_quad) == 0, f"m_kappa != 0 for quadratic U: {sp.expand(m_kappa_quad)}"
print("OK: m_kappa = 0 for every quadratic U")

OK: m_kappa = -((hbar kappa)^2/24) lp^3 U'''(x) + O(kappa^4)
OK: m_kappa = 0 for every quadratic U


## 3. The bound on $|m_\kappa|$ on a grid

$U(x)=-\cos x$, so $U'''(x)=-\sin x$ and $\sup|U'''|=1$. Grid of $256\times256$ points on $[-4\pi,4\pi)^2$, with $\hbar=m=1$.

In [3]:
N = 256
L = 4 * np.pi
grid = np.linspace(-L, L, N, endpoint=False)
X, LP = np.meshgrid(grid, grid, indexing='ij')
HBAR = 1.0
M_MASS = 1.0


def U_num(x):
    return -np.cos(x)


def Uprime_num(x):
    return np.sin(x)


SUP_UPPP = 1.0  # U'''(x) = -sin x

for kv in (0.4, 0.2, 0.1, 0.05):
    hkv = HBAR * kv
    m_k = (U_num(X - hkv * LP / 2) - U_num(X + hkv * LP / 2)) / hkv \
        + LP * Uprime_num(X)
    bound = (hkv**2 / 24) * np.abs(LP)**3 * SUP_UPPP
    assert np.all(np.abs(m_k) <= bound + 1e-12), \
        f"|m_kappa| exceeds the (hk)^2/24 |lp|^3 sup|U'''| bound at kappa={kv}"
print("OK: |m_kappa| <= ((hbar kappa)^2/24)|lambda_p|^3 sup|U'''| on the grid, "
      "for kappa = 0.4, 0.2, 0.1, 0.05  (U = -cos x)")

OK: |m_kappa| <= ((hbar kappa)^2/24)|lambda_p|^3 sup|U'''| on the grid, for kappa = 0.4, 0.2, 0.1, 0.05  (U = -cos x)


## 4. Convergence of solutions at rate $\kappa^2$

The equation $i\,\partial_t\rho=\bigl[\tfrac1m\partial_x\partial_{\lambda_p}+V_\kappa\bigr]\rho$ is evolved by Strang split-step Fourier, with $V_\kappa=\frac{1}{\hbar\kappa}\bigl[U(u)-U(v)\bigr]$ for $\kappa>0$ and $V_0=-\lambda_pU'(x)$ (Liouville). Initial state: a normalized Gaussian. Time step $2\cdot10^{-3}$, 500 steps, final time $t=1$. The cell prints the distance to the Liouville solution for $\kappa=0.4,\,0.2,\,0.1,\,0.05$ and the observed rates $\log_2$ of successive ratios, and asserts that the distances decrease, that every rate exceeds $1.6$, and that the distance at $\kappa=0.05$ is below $10^{-2}$.

In [4]:
kx = 2 * np.pi * np.fft.fftfreq(N, d=2 * L / N)
KX, KL = np.meshgrid(kx, kx, indexing='ij')
DT = 2e-3
STEPS = 500  # T = 1.0
kinetic_phase = np.exp(1j * DT * KX * KL / M_MASS)  # e^{-i dt * (-kx*kl/m)}

rho0 = np.exp(-(X**2 + LP**2) / 2).astype(complex)
rho0 /= np.linalg.norm(rho0)


def potential(kv):
    if kv == 0.0:
        return -LP * Uprime_num(X)
    hkv = HBAR * kv
    return (U_num(X - hkv * LP / 2) - U_num(X + hkv * LP / 2)) / hkv


def evolve(kv):
    half = np.exp(-0.5j * DT * potential(kv))
    rho = rho0.copy()
    for _ in range(STEPS):
        rho = half * rho
        rho = np.fft.ifft2(kinetic_phase * np.fft.fft2(rho))
        rho = half * rho
    return rho


rho_liouville = evolve(0.0)
kappas = [0.4, 0.2, 0.1, 0.05]
errs = []
for kv in kappas:
    err = np.linalg.norm(evolve(kv) - rho_liouville)
    errs.append(err)
    print(f"   kappa={kv:<5} ||rho_kappa(T) - rho_0(T)||_2 = {err:.3e}")

for e1, e2 in zip(errs, errs[1:]):
    assert e2 < e1, "solution error not decreasing as kappa -> 0"
rates = [np.log2(e1 / e2) for e1, e2 in zip(errs, errs[1:])]
print(f"   observed rates under kappa-halving: {[f'{r:.2f}' for r in rates]} (expected ~2)")
for r in rates:
    assert r > 1.6, f"convergence rate {r:.2f} below the predicted O(kappa^2)"
assert errs[-1] < 1e-2, f"solution error at kappa=0.05 too large: {errs[-1]:.3e}"
print("OK: the solutions converge to the Liouville solution at rate O(kappa^2)")

   kappa=0.4   ||rho_kappa(T) - rho_0(T)||_2 = 5.256e-03


   kappa=0.2   ||rho_kappa(T) - rho_0(T)||_2 = 1.317e-03


   kappa=0.1   ||rho_kappa(T) - rho_0(T)||_2 = 3.293e-04


   kappa=0.05  ||rho_kappa(T) - rho_0(T)||_2 = 8.235e-05
   observed rates under kappa-halving: ['2.00', '2.00', '2.00'] (expected ~2)
OK: the solutions converge to the Liouville solution at rate O(kappa^2)


In [5]:
# Every assert above has passed if this cell runs.
print('C7: all checks passed.')

C7: all checks passed.
